# Fit PatchCore on MVTec AD (bottle, screw)

Generated by `cv/build_notebook.py` — edit the source modules, not this notebook.

**Runtime → Change runtime type → GPU (T4)**, then *Run all*. Output: `patchcore_artifacts.zip`
containing `models/<category>/weights/torch/model.pt` and `reports/raw/{scores.csv, maps_*.npz}`.
Unzip it into the repo root.

In [ ]:
!pip install -q anomalib==2.7.0 huggingface_hub
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to GPU"

In [ ]:
import os; os.makedirs('cv', exist_ok=True)

In [ ]:
%%writefile cv/__init__.py

In [ ]:
%%writefile cv/download_mvtec.py
"""Download MVTec AD categories.

anomalib's built-in MVTec URL (mydrive.ch) returns 404 as of 2026-10, so we pull
from a Hugging Face mirror that keeps the original folder layout:
    <root>/<category>/{train/good, test/<defect_type>, ground_truth/<defect_type>}

MVTec AD is licensed CC BY-NC-SA 4.0 (see <root>/license.txt).

Usage: python -m cv.download_mvtec [--root data/mvtec] [--categories bottle screw] [--foreign]
"""
from __future__ import annotations

import argparse
from pathlib import Path

REPO_ID = "foersben/mvtec-ad"
DEFAULT_CATEGORIES = ("bottle", "screw")

# Official MVTec AD test-set sizes; used to detect partial downloads.
EXPECTED_TEST_COUNTS = {"bottle": 83, "screw": 160}


def download(root: Path, categories=DEFAULT_CATEGORIES) -> Path:
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        local_dir=root,
        allow_patterns=[f"{c}/**" for c in categories] + ["license.txt"],
    )
    for c in categories:
        n = len(list((root / c / "test").glob("*/*.png")))
        expected = EXPECTED_TEST_COUNTS.get(c)
        if expected is not None and n != expected:
            raise RuntimeError(f"{c}: found {n} test images, expected {expected}")
        print(f"{c}: {n} test images OK")
    return root


FOREIGN_CATEGORIES = ("cable", "hazelnut", "metal_nut", "toothbrush")


def download_foreign(out: Path = Path("data/foreign"), per_category: int = 3) -> Path:
    """Held-out photos of objects we have no model for, used to test the out-of-distribution guard.
    They are never used for calibration."""
    import shutil
    from huggingface_hub import hf_hub_download

    out.mkdir(parents=True, exist_ok=True)
    for cat in FOREIGN_CATEGORIES:
        for i in range(per_category):
            src = hf_hub_download(REPO_ID, f"{cat}/test/good/{i:03d}.png", repo_type="dataset")
            shutil.copy(src, out / f"{cat}_{i:03d}.png")
    return out


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--root", type=Path, default=Path("data/mvtec"))
    p.add_argument("--categories", nargs="+", default=list(DEFAULT_CATEGORIES))
    p.add_argument("--foreign", action="store_true", help="also fetch held-out foreign-object photos")
    args = p.parse_args()
    download(args.root, args.categories)
    if args.foreign:
        print("foreign photos ->", download_foreign())

In [ ]:
%%writefile cv/patchcore.py
"""PatchCore fit / export / scoring — shared by the Colab notebook and local code.

Post-processing is disabled on purpose: anomalib's post-processor min-max
normalises and clamps scores using thresholds it picks itself. We want raw
anomaly scores and do our own calibration (cv/calibrate.py), so every number
we report comes from one place.

Layout produced by export:   <models_dir>/<category>/weights/torch/model.pt
Layout produced by scoring:  <out_dir>/scores.csv, <out_dir>/maps_<category>.npz
"""
from __future__ import annotations

import csv
import os
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

MAP_SIZE = 256  # anomaly maps are stored at this resolution (float16)


@dataclass(frozen=True)
class ImageRecord:
    category: str
    defect_type: str   # "good" or the MVTec defect folder name
    image: str         # path relative to the data root, posix style
    label: int         # 0 = good, 1 = defective
    mask: str          # ground-truth mask path relative to data root, "" for good


def list_test_images(data_root: Path, category: str) -> list[ImageRecord]:
    records = []
    for img in sorted((data_root / category / "test").glob("*/*.png")):
        defect_type = img.parent.name
        is_good = defect_type == "good"
        mask = "" if is_good else f"{category}/ground_truth/{defect_type}/{img.stem}_mask.png"
        if mask and not (data_root / mask).exists():
            raise FileNotFoundError(f"missing ground-truth mask {mask}")
        records.append(ImageRecord(category, defect_type, img.relative_to(data_root).as_posix(),
                                   0 if is_good else 1, mask))
    if not records:
        raise FileNotFoundError(f"no test images under {data_root / category / 'test'}")
    return records


def fit_and_export(category: str, data_root: Path, models_dir: Path,
                   coreset_sampling_ratio: float = 0.1, num_workers: int = 2) -> Path:
    """Fit a PatchCore memory bank on train/good and export a TorchInferencer model."""
    from anomalib.data import MVTecAD
    from anomalib.deploy import ExportType
    from anomalib.engine import Engine
    from anomalib.models import Patchcore

    if not (data_root / category).is_dir():
        raise FileNotFoundError(f"{data_root / category} not found — run cv.download_mvtec first")
    datamodule = MVTecAD(root=data_root, category=category, num_workers=num_workers)
    model = Patchcore(coreset_sampling_ratio=coreset_sampling_ratio,
                      post_processor=False, evaluator=False, visualizer=False)
    # No progress bar: Lightning's default RichProgressBar proxies stdout/stderr, and
    # PatchCore's coreset tqdm writing through it recurses forever inside Jupyter/Colab.
    # See tests/eval/test_fit_in_jupyter.py.
    engine = Engine(default_root_dir=models_dir / "_work", logger=False, enable_progress_bar=False)
    engine.fit(model=model, datamodule=datamodule)
    path = engine.export(model=model, export_type=ExportType.TORCH, export_root=models_dir / category)
    if path is None or not Path(path).exists():
        raise RuntimeError(f"export produced no model file for {category}")
    return Path(path)


def model_path(models_dir: Path, category: str) -> Path:
    return models_dir / category / "weights" / "torch" / "model.pt"


class PatchCoreScorer:
    """Thin wrapper over anomalib's TorchInferencer returning (score, map)."""

    def __init__(self, path: Path, device: str = "cpu"):
        from anomalib.deploy import TorchInferencer

        # Exported .pt files are pickles; anomalib refuses them unless
        # TRUST_REMOTE_CODE=1. We only load models we exported ourselves, so we
        # opt in for this load alone instead of process-wide.
        previous = os.environ.get("TRUST_REMOTE_CODE")
        os.environ["TRUST_REMOTE_CODE"] = "1"
        try:
            self._inferencer = TorchInferencer(path=path, device=device)
        finally:
            if previous is None:
                del os.environ["TRUST_REMOTE_CODE"]
            else:
                os.environ["TRUST_REMOTE_CODE"] = previous

    def score(self, image_path: Path) -> tuple[float, np.ndarray]:
        import cv2

        pred = self._inferencer.predict(image=image_path)
        score = float(pred.pred_score.detach().cpu().reshape(-1)[0])
        amap = pred.anomaly_map.detach().cpu().numpy().squeeze().astype(np.float32)
        if amap.shape != (MAP_SIZE, MAP_SIZE):
            amap = cv2.resize(amap, (MAP_SIZE, MAP_SIZE), interpolation=cv2.INTER_LINEAR)
        return score, amap


def score_test_set(data_root: Path, models_dir: Path, out_dir: Path,
                   categories=("bottle", "screw"), device: str = "cpu") -> Path:
    """Score every test image; write scores.csv and per-category anomaly maps."""
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for category in categories:
        scorer = PatchCoreScorer(model_path(models_dir, category), device=device)
        maps = {}
        for rec in list_test_images(data_root, category):
            score, amap = scorer.score(data_root / rec.image)
            rows.append({**asdict(rec), "score": score})
            maps[rec.image] = amap.astype(np.float16)
        np.savez_compressed(out_dir / f"maps_{category}.npz", **maps)
        print(f"{category}: scored {len(maps)} images")

    csv_path = out_dir / "scores.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return csv_path

In [ ]:
from pathlib import Path
from cv.download_mvtec import download
DATA, MODELS, RAW = Path("data/mvtec"), Path("models"), Path("reports/raw")
download(DATA)

Fit one memory bank per category (~ a few minutes each on a T4).

In [ ]:
from cv.patchcore import fit_and_export
for category in ["bottle", "screw"]:
    print(category, "->", fit_and_export(category, DATA, MODELS))

Score every test image with the exported model — the same code path the local app uses.

In [ ]:
from cv.patchcore import score_test_set
csv_path = score_test_set(DATA, MODELS, RAW, device="cuda")

Sanity check only — the real evaluation (folds, bands, CIs) runs locally in `cv/evaluate.py`.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
df = pd.read_csv(csv_path)
for cat, g in df.groupby("category"):
    print(f"{cat}: n={len(g)}  image AUROC={roc_auc_score(g.label, g.score):.4f}")

In [ ]:
!rm -rf models/_work
!zip -qr patchcore_artifacts.zip models reports/raw
from google.colab import files
files.download("patchcore_artifacts.zip")